# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

## Data Cleaning

In [48]:
import pandas as pd

train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

train_df.head()

,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,...,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,...,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,...,5,4,3,4,4,4,5,0,0.0,satisfied
3,3,24026,Female,Loyal Customer,25,Business travel,Business,562,2,5,...,2,2,5,3,1,4,2,11,9.0,neutral or dissatisfied
4,4,119299,Male,Loyal Customer,61,Business travel,Business,214,3,3,...,3,3,4,4,3,3,3,0,0.0,satisfied


In [49]:
# Check for missing values
print(train_df.shape[0])
print(train_df.isnull().sum())

print(test_df.shape[0])
print(test_df.isnull().sum())

103904
Unnamed: 0                             0
id                                     0
Gender                                 0
Customer Type                          0
Age                                    0
Type of Travel                         0
Class                                  0
Flight Distance                        0
Inflight wifi service                  0
Departure/Arrival time convenient      0
Ease of Online booking                 0
Gate location                          0
Food and drink                         0
Online boarding                        0
Seat comfort                           0
Inflight entertainment                 0
On-board service                       0
Leg room service                       0
Baggage handling                       0
Checkin service                        0
Inflight service                       0
Cleanliness                            0
Departure Delay in Minutes             0
Arrival Delay in Minutes             310
satisfact

In [50]:
# Drop rows with missing values
train_df = train_df.dropna().reset_index(drop=True)
test_df = test_df.dropna().reset_index(drop=True)

print(train_df.shape[0])
print(train_df.isnull().sum().sum())

print(test_df.shape[0])
print(test_df.isnull().sum().sum())

103594
0
25893
0


In [51]:
# Check for duplicates
duplicate_count_train = train_df.duplicated().sum()
duplicate_count_test = test_df.duplicated().sum()

print(duplicate_count_train)
print(duplicate_count_test)

0
0


## Forward feature selection(call this function to imply selection in each model)

In [52]:
from sklearn.model_selection import cross_val_score
import numpy as np

def forward_feature_selection(X, y, model, cv=5):
    """
    Perform forward feature selection to select the optimal subset of features.
    
    Parameters:
    - X: Feature dataset (DataFrame)
    - y: Target variable (Series)
    - model: Machine learning model instance, e.g., GaussianNB(), LogisticRegression(), etc.
    - cv: Number of cross-validation folds, default is 5
    
    Returns:
    - selected_features: List of selected optimal features
    - best_score: Cross-validation score of the optimal feature subset
    """
    selected_features = []
    remaining_features = list(X.columns)
    best_score = 0

    while remaining_features:
        scores = []
        for feature in remaining_features:
            # Test the performance of the current selected features with the new feature added
            features_to_test = selected_features + [feature]
            # Use cross-validation to evaluate model performance
            score = np.mean(cross_val_score(model, X[features_to_test], y, cv=cv))
            scores.append((score, feature))
        
        # Find the best feature to add
        scores.sort(reverse=True)
        if scores[0][0] > best_score:
            best_score, best_feature = scores[0]
            selected_features.append(best_feature)
            remaining_features.remove(best_feature)
        else:
            break  # Stop if no improvement

    return selected_features, best_score

## One-hot Encoding

In [53]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Define features (X) and target variable (y), ensuring consistent row counts for X and y
X = train_df.drop(columns=['satisfaction', 'id'])  # Remove 'id' and target variable 'satisfaction'
y = train_df['satisfaction']

# Perform One-Hot Encoding on all categorical features
X = pd.get_dummies(X, drop_first=False)

# Define numeric features for standardization
numeric_features = [
    'Age', 'Flight Distance', 'Inflight wifi service', 
    'Departure/Arrival time convenient', 'Ease of Online booking', 
    'Gate location', 'Food and drink', 'Online boarding', 
    'Seat comfort', 'Inflight entertainment', 'On-board service', 
    'Leg room service', 'Baggage handling', 'Checkin service', 
    'Inflight service', 'Cleanliness', 'Departure Delay in Minutes', 
    'Arrival Delay in Minutes'
]

# Extract and standardize numeric features
scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(X[numeric_features])

# Convert standardized numeric features back to DataFrame
X_numeric_scaled_df = pd.DataFrame(X_numeric_scaled, columns=numeric_features)

# Concatenate standardized numeric features with One-Hot encoded categorical features
X_final_scaled = pd.concat([X_numeric_scaled_df, X.drop(columns=numeric_features)], axis=1)

# Display a preview of the final processed dataset
print("\nPreview of X_final_scaled (first 5 rows):")
print(X_final_scaled.head())

print("\nPreview of X (first 5 rows):")
print(X.head())


Preview of X_final_scaled (first 5 rows):
        Age  Flight Distance  Inflight wifi service  \
0 -1.745542        -0.731305               0.203521   
1 -0.951526        -0.956916               0.203521   
2 -0.885358        -0.047454              -0.549571   
3 -0.951526        -0.629028              -0.549571   
4  1.430521        -0.977973               0.203521   

   Departure/Arrival time convenient  Ease of Online booking  Gate location  \
0                           0.616249                0.173716      -1.547312   
1                          -0.695032                0.173716       0.017981   
2                          -0.695032               -0.541118      -0.764666   
3                           1.271890                1.603383       1.583273   
4                          -0.039391                0.173716       0.017981   

   Food and drink  Online boarding  Seat comfort  Inflight entertainment  ...  \
0        1.352401        -0.185632      1.182991                1.2315

## NB Method

In [54]:
from sklearn.naive_bayes import GaussianNB

# Use Naive Bayes for forward feature selection
nb_model = GaussianNB()
selected_features_nb, best_score_nb = forward_feature_selection(X_final_scaled, y, nb_model)
print("Naive Bayes selected features:", selected_features_nb)
print("Naive Bayes best cross-validation score:", best_score_nb)

Naive Bayes selected features: ['Online boarding', 'Checkin service', 'Inflight wifi service', 'Class_Eco', 'Inflight entertainment', 'Type of Travel_Personal Travel', 'Customer Type_disloyal Customer', 'On-board service', 'Leg room service', 'Gender_Male', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Cleanliness']
Naive Bayes best cross-validation score: 0.8784293074997382


In [56]:
# from sklearn.neighbors import KNeighborsClassifier
# 
# # Use KNN for forward feature selection
# knn_model = KNeighborsClassifier(n_neighbors=5, weights='uniform', n_jobs=-1)
# selected_features_knn, best_score_knn = forward_feature_selection(X_final_scaled, y, knn_model)
# print("KNN selected features:", selected_features_knn)
# print("KNN best cross-validation score:", best_score_knn)
